# Representação vetorial de textos

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/correa-ufrrj/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_05/03_representacao_vetorial_de_textos_revisado.ipynb)

Nos notebooks anteriores, comparamos **propriedades numéricas e categóricas entre grupos**. Agora mudamos o objeto da comparação: passaremos a comparar o conteúdo lexical de documentos.

Para isso, um texto precisa ser transformado em uma estrutura sobre a qual possamos calcular. Essa transformação é uma **representação**: ela preserva algumas propriedades do texto e descarta outras.

A pergunta central deste notebook é:

> **Como representar documentos de modo que possamos compará-los sem esquecer o que a representação preserva e o que ela perde?**

Usaremos os mesmos **120 documentos fictícios** da versão revisada da Unidade 5. Como o conjunto ampliado contém metadados e medidas documentais, acrescentaremos a cada registro um pequeno texto sintético, gerado de forma determinística a partir de `tema`, `genero`, `local` e período. Isso permite manter a continuidade com os notebooks anteriores sem tratar os textos como evidência histórica.

## 1. *Bag of Words*: o que preserva e o que perde

A representação *Bag of Words* descreve um documento pelas frequências de seus termos.

Ela preserva principalmente:

- quais termos aparecem;
- quantas vezes aparecem.

Mas descarta, entre outras coisas:

- a ordem das palavras;
- a sintaxe;
- relações de dependência;
- negação;
- ironia;
- parte importante do contexto discursivo.

Por exemplo, `"trabalho não é progresso"` e `"progresso não é trabalho"` recebem as mesmas contagens. Isso não é um erro do algoritmo: é uma **perda deliberada causada pela representação escolhida**.

In [ ]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import os
import subprocess

URL_REPOSITORIO = "https://github.com/correa-ufrrj/disciplina_computacao_aplicada_humanidades_digitais.git"
REPOSITORIO = Path("/dados/disciplina_computacao_aplicada_humanidades_digitais")
PASTA_UNIDADE = REPOSITORIO / "unidade_05"

try:
    import google.colab  # type: ignore
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", "main",
             URL_REPOSITORIO, str(REPOSITORIO)],
            check=True,
        )
    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

### Preparação dos textos didáticos

O conjunto ampliado da unidade possui 120 registros. O notebook procura `dados/documentos.csv`; se ele ainda não existir no ambiente, reproduz o mesmo mecanismo gerador usado nos notebooks anteriores.

Em seguida, acrescenta uma coluna `texto` construída a partir dos metadados. Essa etapa é importante porque deixa explícito que os textos são **sintéticos** e que sua estrutura lexical foi introduzida para fins pedagógicos.

O mecanismo foi desenhado para produzir:

- vocabulário compartilhado por muitos documentos;
- vocabulário associado a cada `tema`;
- algumas palavras associadas a `genero`;
- pequenas diferenças entre `Capital` e `Interior`;
- uma mudança lexical moderada entre os períodos inicial e final.

Essas relações não devem ser “descobertas” como fatos históricos: elas foram deliberadamente incluídas na construção dos dados.

In [ ]:
import math
import re
import unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ARQUIVO = Path("dados/documentos.csv")

def gerar_dados():
    SEED, N = 20261058, 120
    rng = np.random.default_rng(SEED)

    linhas = []
    for ano in range(1890, 1910):
        t = (ano - 1890) / 19
        p_capital = 0.12 + 0.76 * t
        n_capital = int(np.clip(rng.binomial(6, p_capital), 1, 5))
        locais_ano = np.array(
            ["Capital"] * n_capital + ["Interior"] * (6 - n_capital)
        )
        rng.shuffle(locais_ano)
        linhas.extend((ano, local) for local in locais_ano)

    rng.shuffle(linhas)
    anos = np.array([x[0] for x in linhas])
    locais = np.array([x[1] for x in linhas])
    capital = locais == "Capital"

    generos = rng.choice(
        ["notícia", "editorial", "carta"],
        N,
        p=[0.45, 0.30, 0.25],
    )

    temas_possiveis = np.array(
        ["educação", "trabalho", "progresso", "saúde"]
    )
    probs = {
        "notícia": [0.20, 0.25, 0.15, 0.40],
        "editorial": [0.20, 0.30, 0.35, 0.15],
        "carta": [0.40, 0.25, 0.20, 0.15],
    }
    temas = np.array([
        rng.choice(temas_possiveis, p=probs[g])
        for g in generos
    ])

    efeito_genero = {
        "notícia": 0.03,
        "editorial": 0.24,
        "carta": -0.18,
    }
    log_palavras = (
        math.log(650)
        + np.array([efeito_genero[g] for g in generos])
        + np.where(capital, 0.64, 0.0)
        - 0.016 * (anos - 1890)
        + rng.normal(0, 0.24, N)
    )
    palavras = np.maximum(
        140, np.rint(np.exp(log_palavras))
    ).astype(int)

    especiais = rng.choice(N, 3, replace=False)
    palavras[especiais] = np.rint(
        palavras[especiais] * rng.uniform(1.65, 2.05, 3)
    ).astype(int)

    densidade = np.clip(rng.normal(365, 38, N), 270, 470)
    paginas = np.maximum(
        1, np.rint(palavras / densidade)
    ).astype(int)

    taxa = {
        "notícia": 7.0,
        "editorial": 4.7,
        "carta": 5.8,
    }
    pessoas = rng.poisson(
        palavras / 1000 * np.array([taxa[g] for g in generos])
    )

    df = pd.DataFrame({
        "id_documento": [f"R{i:03d}" for i in range(1, N + 1)],
        "ano": anos,
        "genero": generos,
        "local": locais,
        "tema": temas,
        "palavras": palavras,
        "paginas": paginas,
        "pessoas": pessoas,
    })

    ARQUIVO.parent.mkdir(exist_ok=True)
    df.to_csv(ARQUIVO, index=False)
    return df

if ARQUIVO.exists():
    dados = pd.read_csv(ARQUIVO)
    print("Dados carregados de:", ARQUIVO)
else:
    dados = gerar_dados()
    print("Dados reproduzidos em:", ARQUIVO)

# Vocabulários sintéticos usados para gerar textos didáticos.
vocabulario_comum = [
    "debate", "sociedade", "questão", "público",
    "mudança", "vida", "cidade", "discussão",
]

vocabulario_tema = {
    "educação": [
        "escola", "ensino", "professor", "aluno",
        "instrução", "leitura", "aprendizagem",
    ],
    "trabalho": [
        "trabalho", "ofício", "operário", "salário",
        "jornada", "produção", "emprego",
    ],
    "progresso": [
        "progresso", "modernização", "indústria",
        "técnica", "desenvolvimento", "inovação", "futuro",
    ],
    "saúde": [
        "saúde", "doença", "higiene", "hospital",
        "médico", "prevenção", "tratamento",
    ],
}

vocabulario_genero = {
    "notícia": ["informação", "ocorreu", "registro", "relato"],
    "editorial": ["opinião", "argumento", "defesa", "posição"],
    "carta": ["escrevo", "pedido", "lembrança", "correspondência"],
}

vocabulario_local = {
    "Capital": ["avenida", "centro", "imprensa", "instituição"],
    "Interior": ["distrito", "comunidade", "região", "localidade"],
}

vocabulario_periodo = {
    "inicial": ["ordem", "costume", "reforma"],
    "final": ["transformação", "novidade", "expansão"],
}

def construir_texto(linha):
    periodo = "inicial" if linha["ano"] <= 1899 else "final"

    termos = (
        vocabulario_comum[:4]
        + vocabulario_tema[linha["tema"]]
        + vocabulario_genero[linha["genero"]]
        + vocabulario_local[linha["local"]]
        + vocabulario_periodo[periodo]
    )

    # Repetições deliberadas tornam o tema dominante visível na representação.
    termos += vocabulario_tema[linha["tema"]][:3]
    termos += vocabulario_comum[4:7]

    return " ".join(termos)

dados["texto"] = dados.apply(construir_texto, axis=1)

print("Registros:", len(dados))
display(
    dados[[
        "id_documento", "ano", "genero",
        "local", "tema", "texto"
    ]].head(3)
)

A etapa seguinte é a **tokenização**. Ela transforma a sequência de caracteres em unidades que trataremos como termos.

Aqui adotaremos regras simples e explícitas:

1. converter para minúsculas;
2. remover acentos;
3. manter apenas sequências de letras;
4. remover um pequeno conjunto de *stopwords*.

Outra regra produziria outro vocabulário e, portanto, outra matriz. A representação não é independente das decisões de pré-processamento.

Preservamos `id_documento` e `texto` para que qualquer resultado quantitativo possa ser reconduzido ao registro de origem.

In [ ]:
stopwords = {
    "a", "e", "o", "de", "do", "da",
    "em", "nas", "também",
}

def tokenizar(texto):
    normalizado = unicodedata.normalize(
        "NFKD", texto.lower()
    )
    sem_acentos = "".join(
        c for c in normalizado
        if not unicodedata.combining(c)
    )
    return [
        t
        for t in re.findall(r"[a-z]+", sem_acentos)
        if t not in stopwords
    ]

dados["tokens"] = dados["texto"].map(tokenizar)

display(
    dados[[
        "id_documento", "tema", "tokens"
    ]].head(3)
)

## 2. Matriz documento-termo

Depois da tokenização, podemos representar cada documento por suas contagens.

Se $c(t,d)$ é o número de ocorrências do termo $t$ no documento $d$, definimos:

$$
X_{d,t}=c(t,d).
$$

A matriz documento-termo tem:

- **linhas**: documentos;
- **colunas**: termos;
- **células**: frequências dos termos nos documentos.

Um valor zero significa somente que o termo não apareceu naquele documento **segundo as regras adotadas**. Não significa que o conceito correspondente seja historicamente irrelevante ou completamente ausente em sentido semântico.

In [ ]:
contagens = [
    Counter(tokens)
    for tokens in dados["tokens"]
]

matriz_contagens = (
    pd.DataFrame(
        contagens,
        index=dados["id_documento"],
    )
    .fillna(0)
    .astype(int)
)

matriz_contagens.index.name = "id_documento"
vocabulario = sorted(matriz_contagens.columns)

print("Dimensões:", matriz_contagens.shape)
print("Vocabulário:", len(vocabulario), "termos")
display(matriz_contagens.iloc[:8, :12])

In [ ]:
recorte = matriz_contagens.iloc[:12, :14]

plt.figure(figsize=(11, 6))
plt.imshow(recorte.to_numpy(), aspect="auto")
plt.xticks(
    range(recorte.shape[1]),
    recorte.columns,
    rotation=75,
    ha="right",
)
plt.yticks(
    range(recorte.shape[0]),
    recorte.index,
)
plt.colorbar(label="Contagem")
plt.title("Recorte da matriz documento-termo")
plt.tight_layout()
plt.show()

A matriz torna os documentos comparáveis porque todos passam a ser descritos no **mesmo espaço de termos**.

Entretanto, um problema aparece: palavras muito comuns na coleção podem receber contagens altas em muitos documentos e dominar algumas comparações. Para reduzir esse efeito, introduziremos uma ponderação que leva em conta não apenas a frequência no documento, mas também a distribuição do termo pela coleção.

## 3. TF, DF, IDF e TF-IDF

Neste notebook:

- **TF** (*term frequency*) será a contagem bruta do termo no documento;
- **DF** (*document frequency*) será o número de documentos em que o termo aparece;
- **IDF** (*inverse document frequency*) atribuirá maior peso a termos presentes em menos documentos.

Usaremos a forma suavizada:

$$
df(t)
=
\sum_{d=1}^{N}
\mathbf{1}(X_{d,t}>0),
$$

$$
idf(t)
=
\log\left(
\frac{N+1}{df(t)+1}
\right)+1.
$$

Finalmente:

$$
tfidf(t,d)
=
X_{d,t}\times idf(t).
$$

Assim, um termo recebe peso alto quando aparece com frequência no documento e, ao mesmo tempo, é relativamente raro na coleção.

A suavização, a base do logaritmo e eventuais normalizações variam entre implementações. Registrar essas escolhas faz parte da documentação do método.

In [ ]:
N = len(matriz_contagens)

frequencia_documento = (
    matriz_contagens.gt(0).sum(axis=0)
)

idf = (
    (N + 1) / (frequencia_documento + 1)
).map(math.log) + 1

matriz_tfidf = matriz_contagens.mul(
    idf, axis=1
)

documento_exemplo = dados.loc[
    dados["tema"].eq("educação"),
    "id_documento",
].iloc[0]

comparacao_pesos = pd.DataFrame({
    "contagem": matriz_contagens.loc[documento_exemplo],
    "df": frequencia_documento,
    "idf": idf,
    "tfidf": matriz_tfidf.loc[documento_exemplo],
}).sort_values(
    "tfidf",
    ascending=False,
)

print("Documento de exemplo:", documento_exemplo)
display(
    dados.loc[
        dados["id_documento"].eq(documento_exemplo),
        ["id_documento", "ano", "genero", "local", "tema", "texto"],
    ]
)

display(comparacao_pesos.head(12).round(3))

Um peso TF-IDF alto não significa automaticamente que o termo seja o “tema verdadeiro” do documento ou que possua maior importância histórica.

Ele significa algo mais restrito:

> **sob esta representação, o termo combina presença no documento com raridade relativa na coleção.**

Esse ponto é especialmente importante porque o IDF é **relativo ao corpus**. Se acrescentarmos ou retirarmos documentos, os valores de DF e IDF podem mudar. Portanto, o peso de um termo não é uma propriedade isolada da palavra; ele emerge da relação entre o documento e a coleção usada como referência.

## 4. Comparar grupos textuais

A lógica de agrupamento da Unidade 5 também pode ser aplicada às representações textuais.

Em vez de perguntar somente se dois documentos são semelhantes, podemos perguntar:

> **que termos recebem pesos diferentes entre dois grupos de documentos?**

Usaremos dois períodos:

- `1890–1899`;
- `1900–1909`.

Essa divisão é declarada **antes** de inspecionar os pesos. Para cada período, calcularemos a média do TF-IDF de cada termo e depois o contraste entre os grupos.

Se $\bar{w}_{1,t}$ e $\bar{w}_{2,t}$ são os pesos médios do termo $t$ nos dois períodos, definimos:

$$
C_t
=
\bar{w}_{1,t}
-
\bar{w}_{2,t}.
$$

Valores positivos favorecem o primeiro período; valores negativos favorecem o segundo.

Como os textos são sintéticos, esperamos encontrar parte da estrutura lexical que foi deliberadamente introduzida na geração. O objetivo é compreender o método, não produzir uma descoberta histórica.

In [ ]:
periodo = pd.Series(
    np.where(
        dados["ano"].to_numpy() <= 1899,
        "1890–1899",
        "1900–1909",
    ),
    index=dados["id_documento"],
    name="periodo",
)

media_tfidf_periodo = matriz_tfidf.groupby(
    periodo,
    observed=True,
).mean()

contraste_periodos = (
    media_tfidf_periodo.loc["1890–1899"]
    - media_tfidf_periodo.loc["1900–1909"]
)

termos_iniciais = contraste_periodos.nlargest(8)
termos_finais = contraste_periodos.nsmallest(8)

tabela_contrastes = pd.concat([
    termos_iniciais.rename("contraste")
    .rename_axis("termo")
    .reset_index()
    .assign(sentido="maior em 1890–1899"),

    termos_finais.rename("contraste")
    .rename_axis("termo")
    .reset_index()
    .assign(sentido="maior em 1900–1909"),
], ignore_index=True)

display(
    tabela_contrastes[
        ["termo", "contraste", "sentido"]
    ].round(3)
)

O contraste identifica termos cuja representação média difere entre os períodos. Ele **não explica por que** essa diferença existe.

Em dados reais, a interpretação deveria retornar aos documentos para verificar:

- em que contextos os termos aparecem;
- se têm o mesmo sentido nos diferentes documentos;
- se a diferença resulta de tema, gênero, autoria, fonte ou seleção;
- se mudanças de vocabulário correspondem realmente a mudanças conceituais.

Neste conjunto sintético, podemos fazer esse retorno aos casos para verificar se a representação recupera as relações que foram incorporadas no mecanismo gerador.

In [ ]:
termos_para_inspecionar = (
    tabela_contrastes["termo"]
    .head(4)
    .tolist()
)

for termo in termos_para_inspecionar:
    exemplos = dados[
        dados["tokens"].apply(
            lambda tokens: termo in tokens
        )
    ][[
        "id_documento",
        "ano",
        "genero",
        "local",
        "tema",
        "texto",
    ]].head(3)

    print("\nTERMO:", termo)
    display(exemplos)

## 5. O que aprendemos sobre representação

Ao final deste notebook, a cadeia de transformação pode ser resumida como:

$$
\text{texto}
\rightarrow
\text{tokens}
\rightarrow
\text{vocabulário}
\rightarrow
\text{matriz documento-termo}
\rightarrow
\text{TF-IDF}.
$$

Cada seta contém decisões metodológicas.

A representação vetorial permite calcular diferenças e semelhanças, mas não elimina a necessidade de interpretar os documentos. Em particular:

- uma representação seleciona certas propriedades;
- toda representação perde informação;
- TF-IDF depende da coleção de referência;
- termos com pesos altos não são automaticamente mais importantes historicamente;
- resultados agregados entre grupos devem ser reconduzidos aos casos.

No próximo notebook, utilizaremos essas representações para calcular medidas de similaridade entre documentos.

## U05-A04 — Atividade integrada — matriz documentada

**Modalidade:** dupla. **Tempo:** 35 minutos.

1. documente as regras de tokenização e as *stopwords* utilizadas;
2. construa a matriz documento-termo;
3. escolha dois documentos e compare contagens e pesos TF-IDF;
4. escolha um critério de agrupamento (`periodo`, `tema`, `genero` ou `local`);
5. identifique pelo menos dois termos distintivos entre dois grupos;
6. retorne aos documentos em que esses termos aparecem;
7. registre uma interpretação possível e pelo menos uma perda da representação.

**Regra de tokenização:** Escreva aqui.

**Documentos comparados:** Escreva aqui.

**Critério de agrupamento:** Escreva aqui.

**Termos distintivos:** Escreva aqui.

**Casos examinados:** Escreva aqui.

**Interpretação e limites:** Escreva aqui.

Leve a matriz e suas regras ao Notebook 04. A partir delas, calcularemos medidas de similaridade e compararemos o que cada medida revela.